# 02 · ETL e Integração SIH + CNES

**Objetivo:** Carregar os dados traduzidos do SIH e CNES, filtrar internações por IAM (CID I21), realizar limpeza e integrar as duas bases em uma base de modelagem unificada.

**Inputs:**
- `data/interim/sih_*_traduzido.csv` — registros de AIH do SIH com colunas `_DESC` de códigos traduzidos
- `data/interim/cnes_*_traduzido.csv` — tabelas do CNES (ST, LT, EQ, SR, HB) com colunas `_DESC` traduzidas

**Outputs gerados:**
- `data/interim/sih_iam.csv` — internações por IAM (base limpa SIH)
- `data/interim/cnes_hospitais.csv` — base mestre de hospitais (CNES consolidado)
- `data/processed/base_modelagem.csv` — base final SIH × CNES pronta para modelagem


## 0. Configuração do Ambiente

In [1]:
import pandas as pd
import numpy as np
import os
import json
from pathlib import Path
import sys
import gc

In [2]:
# Adiciona a raiz do projeto ao sys.path para importar src/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(f"Raiz do projeto: {ROOT}")

Raiz do projeto: /home/carolina/Documents/TCC Documentos/TCC


In [3]:
pd.set_option('display.float_format', '{:.2f}'.format)

# ── Caminhos ───────────────────────────────────────────────────────────
# Os CSVs de entrada agora são os arquivos *traduzidos* gerados pelo
# notebook 01_data_collection.ipynb e salvos em data/interim/
INTERIM   = Path(ROOT, 'data', 'interim')    # traduzidos (input) e outputs do ETL
PROCESSED = Path(ROOT, 'data', 'processed')
EXTERNAL  = Path(ROOT, 'data', 'external')

INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print("Caminhos configurados:")
print(f"  Entrada: {INTERIM}")
print(f"  Saída  : {PROCESSED}")

Caminhos configurados:
  Entrada: /home/carolina/Documents/TCC Documentos/TCC/data/interim
  Saída  : /home/carolina/Documents/TCC Documentos/TCC/data/processed


## 1. Carregamento e Padronização do SIH

### 1.1 Leitura dos arquivos

Concatenamos **todos** os arquivos CSV disponíveis em `data/input/SIH/`.
As colunas já foram traduzidas e renomeadas na etapa anterior (Notebook 01b).


In [4]:
# Lê os CSVs traduzidos do SIH (padrão: sih_*_traduzido.csv)
arquivos_sih = sorted(INTERIM.glob('sih_*_traduzido.csv'))
print(f"Arquivos SIH traduzidos encontrados: {len(arquivos_sih)}")
for f in arquivos_sih:
    print(f"  {f.name}")

Arquivos SIH traduzidos encontrados: 132
  sih_rdsp1501_traduzido.csv
  sih_rdsp1502_traduzido.csv
  sih_rdsp1503_traduzido.csv
  sih_rdsp1504_traduzido.csv
  sih_rdsp1505_traduzido.csv
  sih_rdsp1506_traduzido.csv
  sih_rdsp1507_traduzido.csv
  sih_rdsp1508_traduzido.csv
  sih_rdsp1509_traduzido.csv
  sih_rdsp1510_traduzido.csv
  sih_rdsp1511_traduzido.csv
  sih_rdsp1512_traduzido.csv
  sih_rdsp1601_traduzido.csv
  sih_rdsp1602_traduzido.csv
  sih_rdsp1603_traduzido.csv
  sih_rdsp1604_traduzido.csv
  sih_rdsp1605_traduzido.csv
  sih_rdsp1606_traduzido.csv
  sih_rdsp1607_traduzido.csv
  sih_rdsp1608_traduzido.csv
  sih_rdsp1609_traduzido.csv
  sih_rdsp1610_traduzido.csv
  sih_rdsp1611_traduzido.csv
  sih_rdsp1612_traduzido.csv
  sih_rdsp1701_traduzido.csv
  sih_rdsp1702_traduzido.csv
  sih_rdsp1703_traduzido.csv
  sih_rdsp1704_traduzido.csv
  sih_rdsp1705_traduzido.csv
  sih_rdsp1706_traduzido.csv
  sih_rdsp1707_traduzido.csv
  sih_rdsp1708_traduzido.csv
  sih_rdsp1709_traduzido.csv
  

In [5]:
# Concatena todos os meses aplicando o filtro IAM DURANTE a leitura
# — lê em chunks de 50 000 linhas
# — filtra imediatamente pelo diagnóstico principal (CID I21)
# — acumula apenas as linhas IAM
# — contabiliza registros antes e depois da filtragem

CID_PREFIXO = "I21"
CHUNK_SIZE  = 50_000

frames = []

# Contadores
total_antes = 0
total_depois = 0

for i, arq in enumerate(arquivos_sih, 1):

    chunks_iam = []
    try:
        reader = pd.read_csv(
            arq,
            dtype=str,
            low_memory=False,
            chunksize=CHUNK_SIZE
        )
        for chunk in reader:
            total_antes += len(chunk)

            chunk["diagnostico_principal_cod"] = (chunk["diagnostico_principal_cod"].astype(str))
            mask = chunk["diagnostico_principal_cod"].str.startswith(CID_PREFIXO)

            iam_chunk = chunk[mask]
            total_depois += len(iam_chunk)
            if not iam_chunk.empty:
                chunks_iam.append(iam_chunk)
            del chunk, iam_chunk, mask
            gc.collect()
        if chunks_iam:
            frames.append(pd.concat(chunks_iam, ignore_index=True))

        del chunks_iam
        gc.collect()

    except Exception as e:

        print(f"  [{i:>3}/{len(arquivos_sih)}] "f"{arq.name} — ERRO: {e}")


df_iam = pd.concat(frames,ignore_index=True)

del frames
gc.collect()

total_removidos = total_antes - total_depois
percentual_mantido = ( total_depois / total_antes * 100 if total_antes > 0 else 0)
percentual_removido = (total_removidos / total_antes * 100 if total_antes > 0 else 0)


print("\n" + "=" * 60)
print("RESUMO DA FILTRAGEM IAM")
print("=" * 60)

print(f"Registros antes da filtragem : {total_antes:,}")
print(f"Registros após filtragem     : {total_depois:,}")
print(f"Registros removidos          : {total_removidos:,}")
print(f"Percentual mantido            : {percentual_mantido:.2f}%")
print(f"Percentual removido           : {percentual_removido:.2f}%")

print("=" * 60)

print(
    f"\nSIH-IAM carregado: "
    f"{df_iam.shape[0]:,} registros × "
    f"{df_iam.shape[1]} colunas"
)

df_iam.head(3)



RESUMO DA FILTRAGEM IAM
Registros antes da filtragem : 28,284,570
Registros após filtragem     : 416,027
Registros removidos          : 27,868,543
Percentual mantido            : 1.47%
Percentual removido           : 98.53%

SIH-IAM carregado: 416,027 registros × 165 colunas


,municipio_gestor_cod,ano_competencia_cod,mes_competencia_cod,especialidade_leito_cod,cnpj_hospital,numero_aih_cod,tipo_aih_cod,cep_paciente,municipio_residencia_cod,data_nascimento,...,tipo_diag_sec_1,tipo_diag_sec_2,tipo_diag_sec_3,tipo_diag_sec_4,tipo_diag_sec_5,tipo_diag_sec_6,tipo_diag_sec_7,tipo_diag_sec_8,tipo_diag_sec_9,FONTE_ORC
0,350000,2015,1,3,57740490000260.0,3514116185327,1,11713110,354100,19571124,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,350000,2015,1,3,46374500011390.0,3515100592289,1,2804050,355030,19520920,...,Preexistente,Preexistente,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,350000,2015,1,3,46374500011390.0,3514120847370,1,2832250,355030,19461225,...,Preexistente,Preexistente,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
#df_iam = pd.read_csv("/home/carolina/Documents/TCC Documentos/TCC/data/interim/sih_iam_modelagem.csv")


In [6]:
# Garante que ano, mês e CNES não tenham o sufixo _cod, para serem mantidos na base de modelagem
# Remove as colunas antigas (que estão 100% nulas por falha no .cnv do DataSUS) e também apaga
# o indicador_obito string, para que possamos renomear o _cod (0/1 numérico) no lugar dele.
cols_to_drop = [c for c in ['ano_competencia', 'mes_competencia', 'cnes', 'codigo_cnes', 'indicador_obito'] if c in df_iam.columns]
df_iam = df_iam.drop(columns=cols_to_drop, errors='ignore')

renames = {
    'ano_competencia_cod': 'ano_competencia',
    'mes_competencia_cod': 'mes_competencia',
    'cnes_cod': 'codigo_cnes',
    'data_internacao_cod': 'data_internacao',
    'data_saida_cod': 'data_saida',
    'indicador_obito_cod': 'indicador_obito'
}
df_iam = df_iam.rename(columns=renames)

df_iam['sexo'] = df_iam['sexo'].replace({'Ignorado': 'Feminino'})


In [7]:
df_iam.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 416027 entries, 0 to 416026
Columns: 162 entries, municipio_gestor_cod to FONTE_ORC
dtypes: object(162)
memory usage: 514.2+ MB


In [8]:
# Testa quais colunas podem ser convertidas para número
# sem alterar o DataFrame original

resultado_numerico = []

for col in df_iam.columns:
    convertido = pd.to_numeric(df_iam[col], errors="coerce")
    
    # quantidade de valores originalmente não nulos
    n_validos = df_iam[col].notna().sum()
    
    # quantidade que conseguiu virar número
    n_numericos = convertido.notna().sum()
    
    resultado_numerico.append({
        "coluna": col,
        "nao_nulos": n_validos,
        "numericos": n_numericos,
        "percentual_numerico": (
            n_numericos / n_validos * 100
            if n_validos > 0 else 0
        )
    })

teste_numerico = pd.DataFrame(resultado_numerico)

teste_numerico.sort_values(
    "percentual_numerico",
    ascending=False
).head(50)

,coluna,nao_nulos,numericos,percentual_numerico
0,municipio_gestor_cod,416027,416027,100.00
1,ano_competencia,416027,416027,100.00
2,mes_competencia,416027,416027,100.00
3,especialidade_leito_cod,416027,416027,100.00
4,cnpj_hospital,349322,349322,100.00
5,numero_aih_cod,416027,416027,100.00
6,tipo_aih_cod,416027,416027,100.00
7,cep_paciente,416027,416027,100.00
8,municipio_residencia_cod,416027,416027,100.00
9,data_nascimento,416027,416027,100.00


### 1.2 Tratamento de Tipos das Colunas
Aplica tipagem explícita de acordo com a regra de negócios:
- `data_` → datetime
- `valor_`, `quantidade_`, `diarias_`, etc. → numérico (float)
- Demais (incluindo `mes_`, `ano_` e IDs) → mantidos como texto seguro
- A limpeza de `000` -> nulo é aplicada apenas nas colunas que **não** terminam em `_cod`.

In [9]:
# 1. Datas (Data)
date_cols = [c for c in df_iam.columns if c.startswith('data_')]
for col in date_cols:
    df_iam[col] = pd.to_datetime(df_iam[col], errors='coerce')

# 2. Numéricas (Float/Int)
num_keywords = ['valor_', 'quantidade_', 'diarias_', 'uti_', 'peso_', 'idade', 'permanencia']
num_cols = [c for c in df_iam.columns if any(kw in c for kw in num_keywords) and c not in date_cols]
for col in num_cols:
    df_iam[col] = pd.to_numeric(df_iam[col], errors='coerce')

# 3. Limpeza de nulos ocultos ('000' -> pd.NA) apenas em Descrições e Strings puras
# Exclui qualquer coluna que termine em _cod para preservar os códigos originais do SUS.
str_cols = df_iam.select_dtypes(include='object').columns
cols_para_limpar = [c for c in str_cols if not c.endswith('_cod')]

df_iam[cols_para_limpar] = (
    df_iam[cols_para_limpar]
    .replace(["", "0000", "000"], pd.NA)
    .infer_objects(copy=False)
)

print(f"Datas formatadas: {len(date_cols)}")
print(f"Numéricos formatados: {len(num_cols)}")
print(f"Colunas de texto limpas (excluindo os códigos): {len(cols_para_limpar)}")


/tmp/ipykernel_933589/434750036.py:19: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(["", "0000", "000"], pd.NA)


Datas formatadas: 4
Numéricos formatados: 42
Colunas de texto limpas (excluindo os códigos): 73


In [16]:
pct_nulos = df_iam.isnull().mean() * 100
print(pct_nulos.sort_values(ascending=False))

cid_notificacao_cod         100.00
numero_processo             100.00
diagnostico_principal       100.00
uti_mes_total               100.00
municipio_estabelecimento   100.00
                             ...  
sequencial_aih5               0.00
raca_cor                      0.00
etnia                         0.00
cid_morte                     0.00
tipo_financiamento            0.00
Length: 162, dtype: float64


#### Tratamento de Valores Ausentes
Campos administrativos do DataSUS frequentemente chegam como strings vazias ou `"0000"`.
Substituímos apenas **colunas de texto** (object), preservando zeros em variáveis
numéricas legítimas (ex.: `indicador_obito=0` = alta; `uti_mes_total=0` = sem UTI).

In [17]:
# ── Remoção de colunas inutilizáveis ─────────────────────────────────────────
# Colunas 100% nulas (sem informação alguma)
limite = 70  # %
cols_remover = pct_nulos[pct_nulos > limite].index
df_iam = df_iam.drop(columns=cols_remover)
print(f'Removidas {len(cols_remover)} colunas com mais de {limite}% de nulos')


# Filtro de idade: manter apenas adultos (>= 18 anos)
#    IAM pediátrico é evento raro e biologicamente distinto — excluído do escopo
df_iam['idade'] = pd.to_numeric(df_iam['idade'], errors='coerce')
n_antes = len(df_iam)
df_iam = df_iam[df_iam['idade'] >= 18].copy()
print(f'Registros pediátricos removidos (idade < 18): {n_antes - len(df_iam)}')
print(f'Base SIH-IAM adultos: {len(df_iam)} registros × {df_iam.shape[1]} colunas')

Removidas 41 colunas com mais de 70% de nulos
Registros pediátricos removidos (idade < 18): 660
Base SIH-IAM adultos: 415367 registros × 121 colunas


In [18]:
# Resumo de completude
nulls = pd.DataFrame({
    "qtd_nulos":  df_iam.isnull().sum(),
    "perc_nulos": df_iam.isnull().mean() * 100
}).sort_values("perc_nulos", ascending=False)

nulls[nulls["qtd_nulos"] > 0]

,qtd_nulos,perc_nulos
diagnostico_secundario_1,222919,53.67
tipo_diag_sec_1,222919,53.67
cnpj_mantenedora,215725,51.94
cnpj_hospital,66493,16.01
inscricao_prenatal,87,0.02


### 1.3 Salvar Base SIH-IAM Intermediária

In [19]:
path_sih_interim = INTERIM / "sih_iam.csv"
df_iam.to_csv(path_sih_interim, index=False)
print(f"SIH-IAM Completo (c/ códigos) salvo em: {path_sih_interim}  ({len(df_iam):,} registros)")

# ── Versão Limpa para Modelagem ──────────────────────────────────────────────
# Remove as colunas _cod para evitar multicolinearidade no modelo preditivo
# Apaga a coluna _cod APENAS se a versão de texto (descrição) sobreviveu à limpeza de nulos
cols_cod = [c for c in df_iam.columns if c.endswith('_cod') and c[:-4] in df_iam.columns]
df_iam_modelagem = df_iam.drop(columns=cols_cod)

path_sih_modelagem = INTERIM / "sih_iam_modelagem.csv"
df_iam_modelagem.to_csv(path_sih_modelagem, index=False)
print(f"SIH-IAM Modelagem (s/ códigos) salvo em: {path_sih_modelagem}  ({len(df_iam_modelagem):,} registros)")


SIH-IAM Completo (c/ códigos) salvo em: /home/carolina/Documents/TCC Documentos/TCC/data/interim/sih_iam.csv  (415,367 registros)
SIH-IAM Modelagem (s/ códigos) salvo em: /home/carolina/Documents/TCC Documentos/TCC/data/interim/sih_iam_modelagem.csv  (415,367 registros)


## 2. Carregamento e Padronização do CNES

Carregamos as cinco tabelas do CNES usando dicionários JSON específicos para cada prefixo.

In [ ]:
df_iam = pd.read_csv("/home/carolina/Documents/TCC Documentos/TCC/data/interim/sih_iam_modelagem.csv")

In [20]:
# Lê os CSVs traduzidos do CNES (padrão: cnes_*_traduzido.csv)
arquivos_cnes = sorted(INTERIM.glob('cnes_*_traduzido.csv'))
print(f"Arquivos CNES traduzidos encontrados: {len(arquivos_cnes)}")
for f in arquivos_cnes:
    print(f"  {f.name}")

Arquivos CNES traduzidos encontrados: 660
  cnes_eq_eqsp1501_traduzido.csv
  cnes_eq_eqsp1502_traduzido.csv
  cnes_eq_eqsp1503_traduzido.csv
  cnes_eq_eqsp1504_traduzido.csv
  cnes_eq_eqsp1505_traduzido.csv
  cnes_eq_eqsp1506_traduzido.csv
  cnes_eq_eqsp1507_traduzido.csv
  cnes_eq_eqsp1508_traduzido.csv
  cnes_eq_eqsp1509_traduzido.csv
  cnes_eq_eqsp1510_traduzido.csv
  cnes_eq_eqsp1511_traduzido.csv
  cnes_eq_eqsp1512_traduzido.csv
  cnes_eq_eqsp1601_traduzido.csv
  cnes_eq_eqsp1602_traduzido.csv
  cnes_eq_eqsp1603_traduzido.csv
  cnes_eq_eqsp1604_traduzido.csv
  cnes_eq_eqsp1605_traduzido.csv
  cnes_eq_eqsp1606_traduzido.csv
  cnes_eq_eqsp1607_traduzido.csv
  cnes_eq_eqsp1608_traduzido.csv
  cnes_eq_eqsp1609_traduzido.csv
  cnes_eq_eqsp1610_traduzido.csv
  cnes_eq_eqsp1611_traduzido.csv
  cnes_eq_eqsp1612_traduzido.csv
  cnes_eq_eqsp1701_traduzido.csv
  cnes_eq_eqsp1702_traduzido.csv
  cnes_eq_eqsp1703_traduzido.csv
  cnes_eq_eqsp1704_traduzido.csv
  cnes_eq_eqsp1705_traduzido.csv
 

In [21]:
import re
def load_cnes_custom(tipo, arquivos_cnes, cnes_vip):
    tipo = tipo.lower()

    # Localiza TODOS os arquivos traduzidos com prefixo cnes_{tipo}_
    arqs = [f for f in arquivos_cnes if f.name.lower().startswith(f'cnes_{tipo}_')]

    if not arqs:
        raise FileNotFoundError(
            f"Nenhum arquivo traduzido cnes_{tipo}_*_traduzido.csv encontrado.\n"
            f"Arquivos disponíveis: {[f.name for f in arquivos_cnes]}"
        )

    dfs = []
    for arq in arqs:
        try:
            reader = pd.read_csv(arq, dtype=str, low_memory=False, chunksize=50_000)
            
            # Puxa o ano e mes do filename apenas uma vez para o arquivo
            match = re.search(r'\D(\d{2})(\d{2})_traduzido', arq.name.lower())
            if match:
                ano_str = match.group(1)
                mes = match.group(2)
                ano = f"20{ano_str}" if int(ano_str) < 50 else f"19{ano_str}"
                competencia_val = f"{ano}{mes}"
            else:
                competencia_val = '000000'
                
            for chunk in reader:
                # OTIMIZAÇÃO OOM: Joga no lixo postinhos e mantem só os hospitais vip!
                chunk['codigo_cnes'] = chunk['codigo_cnes'].astype(str).str.strip().str.zfill(7)
                chunk = chunk[chunk['codigo_cnes'].isin(cnes_vip)]
                
                if not chunk.empty:
                    chunk = chunk.copy() # para evitar warnings de SettingWithCopy
                    chunk['competencia'] = competencia_val
                    chunk['arquivo_origem'] = arq.name
                    dfs.append(chunk)
                    
        except Exception as e:
            print(f"Erro lendo {arq.name}: {e}")

    if not dfs:
        print(f"Aviso: Tabela {tipo.upper()} não continha nenhum hospital de interesse.")
        return pd.DataFrame(columns=['codigo_cnes', 'competencia', 'tipo_cnes', 'arquivo_origem'])
        
    df_final = pd.concat(dfs, ignore_index=True)
    df_final['tipo_cnes'] = tipo.upper()

    return df_final


In [22]:
# Lista de hospitais que internaram infartados (anti-OOM):
cnes_vip = set(df_iam['codigo_cnes'].astype(str).str.strip().str.zfill(7).unique())
print(f"Hospitais VIP (SIH): {len(cnes_vip)}\n")

df_st = load_cnes_custom('st', arquivos_cnes, cnes_vip)
df_lt = load_cnes_custom('lt', arquivos_cnes, cnes_vip)
df_eq = load_cnes_custom('eq', arquivos_cnes, cnes_vip)
df_sr = load_cnes_custom('sr', arquivos_cnes, cnes_vip)
df_hb = load_cnes_custom('hb', arquivos_cnes, cnes_vip)

print(f"\nRegistros carregados por tabela:")
print(f"  ST (Estabelecimentos) : {df_st.shape}")
print(f"  LT (Leitos)           : {df_lt.shape}")
print(f"  EQ (Equipamentos)     : {df_eq.shape}")
print(f"  SR (Serviços)         : {df_sr.shape}")
print(f"  HB (Habilitações)     : {df_hb.shape}")


Hospitais VIP (SIH): 506



/tmp/ipykernel_933589/3184462960.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tipo_cnes'] = tipo.upper()



Registros carregados por tabela:
  ST (Estabelecimentos) : (62552, 259)
  LT (Leitos)           : (552840, 41)
  EQ (Equipamentos)     : (1471830, 43)
  SR (Serviços)         : (2213516, 46)
  HB (Habilitações)     : (427109, 49)


### 2.1 pivot Tabelas CNES 

#### Remoção de Duplicidades (ST)

In [23]:
# Garante uma linha por hospital — mantém o registro mais recente
n_antes = len(df_st)
df_st = df_st.drop_duplicates(subset=['codigo_cnes', 'competencia'], keep='last')
print(f"ST: {n_antes} → {len(df_st)} registros únicos (removidas {n_antes - len(df_st):,} duplicatas)")

ST: 62552 → 62552 registros únicos (removidas 0 duplicatas)


#### Agregação de Leitos (LT)

In [24]:
# Pivot de leitos por tipo
df_lt['quantidade_leitos_existentes'] = pd.to_numeric(
    df_lt['quantidade_leitos_existentes'],
    errors='coerce'
).fillna(0)

df_lt_pivot = df_lt.pivot_table(
    index=['codigo_cnes', 'competencia'],
    columns='tipo_leito',
    values='quantidade_leitos_existentes',
    aggfunc='sum',
    fill_value=0
)

df_lt_pivot.columns = [
    f'leitos_{col}'
    for col in df_lt_pivot.columns
]

df_lt_pivot = df_lt_pivot.reset_index()

#### Agregação de Equipamentos (EQ)

In [25]:
# Pivot de equipamentos
df_eq['quantidade_em_uso'] = pd.to_numeric(
    df_eq['quantidade_em_uso'],
    errors='coerce'
).fillna(0)

df_eq_pivot = df_eq.pivot_table(
    index=['codigo_cnes', 'competencia'],
    columns='codigo_equipamento',
    values='quantidade_em_uso',
    aggfunc='sum',
    fill_value=0
)

df_eq_pivot.columns = [
    f'equip_{col}'
    for col in df_eq_pivot.columns
]

df_eq_pivot = df_eq_pivot.reset_index()

#### Serviços Especializados (SR) 

In [26]:
# Serviços especializados
df_sr['flag_servico'] = 1

df_sr_pivot = df_sr.pivot_table(
    index=['codigo_cnes', 'competencia'],
    columns='codigo_servico_especializado',
    values='flag_servico',
    aggfunc='max',
    fill_value=0
)

df_sr_pivot.columns = [
    f'servico_{col}'
    for col in df_sr_pivot.columns
]

df_sr_pivot = df_sr_pivot.reset_index()

#### Habilitações (HB)

In [27]:
# Habilitações
df_hb['flag_habilitacao'] = 1

df_hb_pivot = df_hb.pivot_table(
    index=['codigo_cnes', 'competencia'],
    columns='codigo_habilitacao',
    values='flag_habilitacao',
    aggfunc='max',
    fill_value=0
)

df_hb_pivot.columns = [
    f'habilitacao_{col}'
    for col in df_hb_pivot.columns
]

df_hb_pivot = df_hb_pivot.reset_index()

In [28]:
print(df_lt_pivot.shape)
print(df_eq_pivot.shape)
print(df_sr_pivot.shape)
print(df_hb_pivot.shape)

(62027, 4)
(62481, 97)
(62339, 64)
(48991, 211)


## 3 Salvar Base Mestre de Hospitais (Interim CNES)

In [29]:
# ── Merge das tabelas CNES em torno da ST (uma linha por hospital) ───────────
#
# ST  : tabela mestre — um registro por hospital (já deduplicada acima)
# LT  : leitos por tipo de leito  → pivotado em df_lt_pivot
# EQ  : equipamentos por código   → pivotado em df_eq_pivot
# SR  : serviços especializados   → pivotado em df_sr_pivot
# HB  : habilitações              → pivotado em df_hb_pivot
#
# Usamos left join para manter todos os hospitais da ST,
# mesmo que não possuam leitos, equipamentos, etc. registrados.

df_cnes_final = df_st.copy()

for df_pivot, nome in [
    (df_lt_pivot, 'LT — Leitos'),
    (df_eq_pivot, 'EQ — Equipamentos'),
    (df_sr_pivot, 'SR — Serviços Especializados'),
    (df_hb_pivot, 'HB — Habilitações'),
]:
    # Garante que a chave está no mesmo formato (string sem espaços)
    df_pivot['codigo_cnes'] = df_pivot['codigo_cnes'].astype(str).str.strip()
    df_cnes_final = pd.merge(df_cnes_final, df_pivot, on=['codigo_cnes', 'competencia'], how='left')
    print(f'  Após merge {nome}: {df_cnes_final.shape}')

# Preenche NaN nas colunas de contagem/flag com 0
# (hospitais sem leitos/equipamentos/serviços registrados ficam como 0, não NaN)
keys_to_drop = ['codigo_cnes', 'competencia']
cols_pivot = (
    list(df_lt_pivot.columns.drop(keys_to_drop, errors='ignore')) +
    list(df_eq_pivot.columns.drop(keys_to_drop, errors='ignore')) +
    list(df_sr_pivot.columns.drop(keys_to_drop, errors='ignore')) +
    list(df_hb_pivot.columns.drop(keys_to_drop, errors='ignore'))
)
df_cnes_final[cols_pivot] = df_cnes_final[cols_pivot].fillna(0)

# Remove colunas auxiliares que vieram das tabelas secundárias e são redundantes
# (arquivo_origem e tipo_cnes existem em cada pivot — causam sufixos _x/_y)
cols_remover_aux = [c for c in df_cnes_final.columns if c.endswith(('_x', '_y'))]
if cols_remover_aux:
    df_cnes_final = df_cnes_final.drop(columns=cols_remover_aux)
    print(f'  Colunas auxiliares duplicadas removidas: {cols_remover_aux}')

print(f'\nBase CNES final: {df_cnes_final.shape[0]:,} hospitais × {df_cnes_final.shape[1]} colunas')


  Após merge LT — Leitos: (62552, 261)
  Após merge EQ — Equipamentos: (62552, 356)
  Após merge SR — Serviços Especializados: (62552, 418)
  Após merge HB — Habilitações: (62552, 627)

Base CNES final: 62,552 hospitais × 627 colunas


### 3.0 Tratamento de Tipos das Colunas do CNES
Tipagem rigorosa para evitar que identificadores (CNPJ, CEP, Códigos) sejam destruídos por conversões automáticas e para garantir que números reais do pivot sejam matemáticos.

In [30]:
# 1. Numéricas (Float/Int)
# Colunas originadas dos pivots (leitos, equipamentos, serviços e habilitações)
num_prefixes = ('leitos_', 'equip_', 'servico_', 'habilitacao_')
num_cols = [c for c in df_cnes_final.columns if c.startswith(num_prefixes)]

for col in num_cols:
    df_cnes_final[col] = pd.to_numeric(df_cnes_final[col], errors='coerce')

# 2. Limpeza de nulos ocultos ('000' -> pd.NA) apenas em Strings puras
# Exclui colunas _cod para manter integridade dos IDs originais do SUS e IBGE
str_cols = df_cnes_final.select_dtypes(include='object').columns
cols_para_limpar = [c for c in str_cols if not c.endswith('_cod')]

df_cnes_final[cols_para_limpar] = (
    df_cnes_final[cols_para_limpar]
    .replace(["", "0000", "000"], pd.NA)
    .infer_objects(copy=False)
)

print(f"Numéricos formatados no CNES: {len(num_cols)}")
print(f"Colunas de texto limpas no CNES (ignorando _cod): {len(cols_para_limpar)}")


/tmp/ipykernel_933589/3141306695.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(["", "0000", "000"], pd.NA)


Numéricos formatados no CNES: 401
Colunas de texto limpas no CNES (ignorando _cod): 167


### 3.1 Limpeza de Colunas com Muitos Nulos

Assim como feito com o SIH, removemos colunas com alto percentual de valores ausentes.
Colunas com `> 70%` de nulos não carregam informação útil para a modelagem e apenas
adicionam ruído e custo computacional.

**Tipos de colunas que tendem a ser removidas:**
- Colunas `_DESC` sem dicionário `.cnv` mapeado (ficam 100% nulas)
- Campos de controle administrativo do CNES nunca preenchidos em SP
- Campos de contrato municipal/estadual que só existem em alguns estados


In [31]:
# ── Inspeção de nulos no CNES após merge ─────────────────────────────────────
pct_nulos_cnes = df_cnes_final.isnull().mean() * 100

# Distribuição por faixa
faixas = {
    '100%'  : (pct_nulos_cnes == 100).sum(),
    '70–99%': ((pct_nulos_cnes >= 70) & (pct_nulos_cnes < 100)).sum(),
    '50–69%': ((pct_nulos_cnes >= 50) & (pct_nulos_cnes < 70)).sum(),
    '1–49%' : ((pct_nulos_cnes > 0)  & (pct_nulos_cnes < 50)).sum(),
    '0%'    : (pct_nulos_cnes == 0).sum(),
}
print(f'Shape antes da limpeza: {df_cnes_final.shape}')
print('\nDistribuição de nulos por coluna:')
for faixa, qtd in faixas.items():
    print(f'  {faixa:8}: {qtd:>4} colunas')

print('\nTop 20 colunas com mais nulos:')
print(pct_nulos_cnes.sort_values(ascending=False).head(20).to_string())


Shape antes da limpeza: (62552, 627)

Distribuição de nulos por coluna:
  100%    :   19 colunas
  70–99%  :   16 colunas
  50–69%  :    7 colunas
  1–49%   :    9 colunas
  0%      :  576 colunas

Top 20 colunas com mais nulos:
data_publicacao_contrato_municipal_cod         100.00
data_publicacao_contrato_estadual_cod          100.00
classificacao_avaliacao_cod                    100.00
data_acreditacao                               100.00
data_avaliacao_pnass                           100.00
numero_contrato_municipal                      100.00
classificacao_avaliacao                        100.00
numero_contrato_estadual                       100.00
servico_apoio_servico_social_proprio           100.00
servico_apoio_farmacia_proprio                 100.00
servico_apoio_esterilizacao_proprio            100.00
servico_apoio_nutricao_proprio                 100.00
servico_apoio_lactario_proprio                 100.00
servico_apoio_banco_leite_proprio              100.00
servico_apoio_l

In [32]:
# ── Remoção de colunas com > 70% de nulos ────────────────────────────────────
LIMITE_NULOS = 70  # mesmo critério aplicado ao SIH

cols_remover_cnes = pct_nulos_cnes[pct_nulos_cnes > LIMITE_NULOS].index.tolist()

df_cnes_final = df_cnes_final.drop(columns=cols_remover_cnes)

print(f'Colunas removidas (> {LIMITE_NULOS}% nulos): {len(cols_remover_cnes)}')
print(f'Shape após limpeza: {df_cnes_final.shape}')
print()

# Resumo de completude após limpeza
nulls_cnes = pd.DataFrame({
    'qtd_nulos' : df_cnes_final.isnull().sum(),
    'perc_nulos': df_cnes_final.isnull().mean() * 100
}).sort_values('perc_nulos', ascending=False)

restantes_com_nulos = nulls_cnes[nulls_cnes['qtd_nulos'] > 0]
print(f'Colunas restantes com algum nulo: {len(restantes_com_nulos)}')
if not restantes_com_nulos.empty:
    display(restantes_com_nulos)


Colunas removidas (> 70% nulos): 35
Shape após limpeza: (62552, 592)

Colunas restantes com algum nulo: 16


,qtd_nulos,perc_nulos
AP01CV07,31791,50.82
AP02CV07,31791,50.82
AP03CV07,31791,50.82
AP04CV07,31791,50.82
AP05CV07,31791,50.82
AP06CV07,31791,50.82
AP07CV07,31791,50.82
codigo_agencia,24882,39.78
conta_corrente,24873,39.76
codigo_banco,24873,39.76


In [33]:
# ── Versão Limpa para Modelagem ──────────────────────────────────────────────
# 1. Criar variáveis macro agrupadas ANTES de deletar as originais numéricas
def macro_natureza(cod):
    if pd.isna(cod):
        return 'Não informado'
    try:
        cod = int(cod)
    except:
        return 'Outro'
    if 2000 <= cod <= 4999:
        return 'Público'
    elif cod in (5045, 5053):
        return 'OS/OSCIP'
    elif cod in (5010, 5029, 5037):
        return 'Privado s/ fins lucrativos'
    elif cod >= 5000:
        return 'Privado c/ fins lucrativos'
    elif 1000 <= cod <= 1999:
        return 'Privado c/ fins lucrativos'
    else:
        return 'Outro'

if 'natureza_juridica_cod' in df_cnes_final.columns:
    df_cnes_final['natureza_juridica_macro'] = df_cnes_final['natureza_juridica_cod'].apply(macro_natureza)
else:
    df_cnes_final['natureza_juridica_macro'] = 'Não informado'

# 2. Remove as colunas _cod para evitar multicolinearidade
# Apaga a coluna _cod APENAS se a versão de texto (descrição) sobreviveu à limpeza
cols_cod_cnes = [c for c in df_cnes_final.columns if c.endswith('_cod') and c[:-4] in df_cnes_final.columns]
df_cnes_modelagem = df_cnes_final.drop(columns=cols_cod_cnes)

path_cnes_modelagem = INTERIM / 'cnes_hospitais_modelagem.csv'
df_cnes_modelagem.to_csv(path_cnes_modelagem, index=False)
print(f'CNES Modelagem (s/ códigos) salvo em: {path_cnes_modelagem}  ({len(df_cnes_modelagem):,} hospitais)')


CNES Interim Completo (c/ códigos) salvo em: /home/carolina/Documents/TCC Documentos/TCC/data/interim/cnes_hospitais.csv  (62,552 hospitais)
CNES Modelagem (s/ códigos) salvo em: /home/carolina/Documents/TCC Documentos/TCC/data/interim/cnes_hospitais_modelagem.csv  (62,552 hospitais)


---
## Resumo do Pipeline ETL

| Etapa | Input | Output | Registros |
|-------|-------|--------|-----------|
| 1. Filtro IAM (SIH) | `data/interim/sih_*_traduzido.csv` | `data/interim/sih_iam_modelagem.csv` | |
| 2. Consolidação CNES | `data/interim/cnes_*_traduzido.csv` | `data/interim/cnes_hospitais_modelagem.csv` | |

> **Próximos passos:** notebook `03_merge_and_eda.ipynb` — análise exploratória e fusão final.